# TensorFlow 模型保存、載入與部署完整指南

<br>
<a href="https://colab.research.google.com/github/markl-a/My-AI-Learning-Notes/blob/main/1.從AI到LLM基礎/4.DL/01.Tensorflow2/6.Model_Saving_and_Deployment.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>
<br>

## 📚 學習目標

本教程將涵蓋：

1. ✅ 模型保存的不同格式（SavedModel, HDF5, Checkpoints）
2. ✅ 模型載入與恢復訓練
3. ✅ 模型轉換（TFLite, ONNX, TF.js）
4. ✅ 模型優化技術
5. ✅ 部署策略與最佳實踐

---

In [ ]:
# 環境設置
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
from tensorflow import keras
import tempfile

print(f"TensorFlow 版本: {tf.__version__}")

# 創建臨時目錄用於保存模型
model_dir = tempfile.mkdtemp()
print(f"模型保存目錄: {model_dir}")

## 1. 訓練一個示例模型

首先，讓我們訓練一個簡單的模型，用於後續的保存和載入演示。

In [ ]:
# 載入 MNIST 數據集
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# 數據預處理
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# 建立模型
def create_model():
    """創建示例模型"""
    model = keras.Sequential([
        keras.layers.Flatten(input_shape=(28, 28)),
        keras.layers.Dense(128, activation='relu', name='hidden_layer'),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(10, activation='softmax', name='output_layer')
    ])
    
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# 創建並訓練模型
model = create_model()
model.summary()

# 訓練模型
history = model.fit(
    x_train, y_train,
    epochs=5,
    validation_split=0.2,
    verbose=1
)

# 評估模型
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\n測試準確率: {test_acc:.4f}")

## 2. 模型保存格式

### 2.1 SavedModel 格式（推薦）

SavedModel 是 TensorFlow 的標準格式，包含：
- 模型架構
- 權重
- 訓練配置
- 優化器狀態

**優點：**
- ✅ 跨平台兼容
- ✅ 支持 TensorFlow Serving
- ✅ 易於轉換到其他格式

In [ ]:
# 保存為 SavedModel 格式
saved_model_path = os.path.join(model_dir, 'saved_model')
model.save(saved_model_path)

print(f"模型已保存到: {saved_model_path}")
print("\n目錄結構:")
!ls -lR {saved_model_path}

# 載入 SavedModel
loaded_model = keras.models.load_model(saved_model_path)

# 驗證載入的模型
loaded_loss, loaded_acc = loaded_model.evaluate(x_test, y_test, verbose=0)
print(f"\n載入模型的測試準確率: {loaded_acc:.4f}")
print(f"原始模型的測試準確率: {test_acc:.4f}")
print(f"準確率是否相同: {np.isclose(loaded_acc, test_acc)}")

### 2.2 HDF5 格式（傳統格式）

HDF5 格式將所有內容保存在單一 .h5 文件中。

**何時使用：**
- 需要單一文件
- 向後兼容舊代碼

In [ ]:
# 保存為 HDF5 格式
h5_model_path = os.path.join(model_dir, 'model.h5')
model.save(h5_model_path)

print(f"模型已保存到: {h5_model_path}")

# 查看文件大小
file_size = os.path.getsize(h5_model_path) / (1024 * 1024)  # MB
print(f"文件大小: {file_size:.2f} MB")

# 載入 HDF5 模型
loaded_h5_model = keras.models.load_model(h5_model_path)

# 驗證
h5_loss, h5_acc = loaded_h5_model.evaluate(x_test, y_test, verbose=0)
print(f"HDF5 模型測試準確率: {h5_acc:.4f}")

### 2.3 Checkpoint 格式（僅保存權重）

Checkpoint 只保存模型權重，不包含架構。

**適用場景：**
- 訓練期間定期保存
- 模型架構已確定

In [ ]:
# 只保存權重
weights_path = os.path.join(model_dir, 'model_weights.h5')
model.save_weights(weights_path)

print(f"權重已保存到: {weights_path}")

# 載入權重（需要先創建相同架構的模型）
new_model = create_model()
new_model.load_weights(weights_path)

# 需要重新編譯
new_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 驗證
weights_loss, weights_acc = new_model.evaluate(x_test, y_test, verbose=0)
print(f"載入權重後的測試準確率: {weights_acc:.4f}")

## 3. 訓練期間自動保存

### 3.1 ModelCheckpoint Callback

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# 創建 checkpoint 目錄
checkpoint_dir = os.path.join(model_dir, 'checkpoints')
os.makedirs(checkpoint_dir, exist_ok=True)

# 定義 checkpoint 文件路徑（包含 epoch 和驗證損失）
checkpoint_path = os.path.join(checkpoint_dir, 'model-{epoch:02d}-{val_loss:.2f}.h5')

# 創建 ModelCheckpoint callback
checkpoint_callback = ModelCheckpoint(
    filepath=checkpoint_path,
    monitor='val_loss',  # 監控指標
    save_best_only=True,  # 只保存最佳模型
    save_weights_only=False,  # 保存完整模型
    mode='min',  # 指標越小越好
    verbose=1
)

# 添加 EarlyStopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

# 訓練新模型
new_model = create_model()
history = new_model.fit(
    x_train, y_train,
    epochs=10,
    validation_split=0.2,
    callbacks=[checkpoint_callback, early_stop],
    verbose=1
)

# 查看保存的 checkpoints
print("\n保存的 checkpoints:")
!ls -lh {checkpoint_dir}

### 3.2 自定義保存策略

In [ ]:
from tensorflow.keras.callbacks import Callback

class CustomSaveCallback(Callback):
    """自定義保存回調"""
    
    def __init__(self, save_dir, save_freq=5):
        super().__init__()
        self.save_dir = save_dir
        self.save_freq = save_freq  # 每 N 個 epoch 保存一次
        os.makedirs(save_dir, exist_ok=True)
    
    def on_epoch_end(self, epoch, logs=None):
        """每個 epoch 結束時調用"""
        if (epoch + 1) % self.save_freq == 0:
            # 保存模型
            save_path = os.path.join(
                self.save_dir,
                f'model_epoch_{epoch+1}_acc_{logs["val_accuracy"]:.4f}.h5'
            )
            self.model.save(save_path)
            print(f"\n模型已保存到: {save_path}")

# 使用自定義 callback
custom_save_dir = os.path.join(model_dir, 'custom_saves')
custom_callback = CustomSaveCallback(custom_save_dir, save_freq=2)

# 訓練（示例，已註解以節省時間）
# model = create_model()
# model.fit(x_train, y_train, epochs=6, validation_split=0.2, callbacks=[custom_callback])

## 4. 模型轉換

### 4.1 轉換為 TensorFlow Lite（移動端/嵌入式）

In [ ]:
# 轉換為 TFLite 格式
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# 基本轉換
tflite_model = converter.convert()

# 保存 TFLite 模型
tflite_path = os.path.join(model_dir, 'model.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

# 比較文件大小
original_size = os.path.getsize(h5_model_path) / 1024  # KB
tflite_size = os.path.getsize(tflite_path) / 1024  # KB

print(f"原始模型大小: {original_size:.2f} KB")
print(f"TFLite 模型大小: {tflite_size:.2f} KB")
print(f"壓縮比例: {original_size/tflite_size:.2f}x")

# 使用 TFLite 模型進行推理
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

# 獲取輸入和輸出張量
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("\n輸入詳情:", input_details[0]['shape'])
print("輸出詳情:", output_details[0]['shape'])

# 測試推理
test_image = x_test[0:1].astype(np.float32)
interpreter.set_tensor(input_details[0]['index'], test_image)
interpreter.invoke()
tflite_prediction = interpreter.get_tensor(output_details[0]['index'])

# 比較結果
original_prediction = model.predict(test_image, verbose=0)
print(f"\n原始模型預測: {np.argmax(original_prediction)}")
print(f"TFLite 模型預測: {np.argmax(tflite_prediction)}")
print(f"實際標籤: {y_test[0]}")

### 4.2 TFLite 量化（進一步壓縮）

In [ ]:
# 動態範圍量化（最簡單）
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
quantized_tflite_model = converter.convert()

# 保存量化模型
quantized_path = os.path.join(model_dir, 'model_quantized.tflite')
with open(quantized_path, 'wb') as f:
    f.write(quantized_tflite_model)

# 比較大小
quantized_size = os.path.getsize(quantized_path) / 1024  # KB

print("模型大小比較:")
print(f"原始 H5: {original_size:.2f} KB")
print(f"TFLite: {tflite_size:.2f} KB ({original_size/tflite_size:.2f}x 壓縮)")
print(f"量化 TFLite: {quantized_size:.2f} KB ({original_size/quantized_size:.2f}x 壓縮)")

# 測試量化模型
interpreter_quant = tf.lite.Interpreter(model_path=quantized_path)
interpreter_quant.allocate_tensors()

input_details_quant = interpreter_quant.get_input_details()
output_details_quant = interpreter_quant.get_output_details()

interpreter_quant.set_tensor(input_details_quant[0]['index'], test_image)
interpreter_quant.invoke()
quantized_prediction = interpreter_quant.get_tensor(output_details_quant[0]['index'])

print(f"\n量化模型預測: {np.argmax(quantized_prediction)}")
print(f"準確性保持: {np.argmax(quantized_prediction) == np.argmax(original_prediction)}")

### 4.3 轉換為 ONNX（跨框架）

In [ ]:
# 安裝 tf2onnx（如果需要）
# !pip install tf2onnx

# 使用命令行工具轉換
onnx_path = os.path.join(model_dir, 'model.onnx')

# 轉換命令（需要先保存 SavedModel）
print("轉換為 ONNX 的命令:")
print(f"python -m tf2onnx.convert --saved-model {saved_model_path} --output {onnx_path}")

# 實際轉換（如果已安裝 tf2onnx）
try:
    import tf2onnx
    !python -m tf2onnx.convert --saved-model {saved_model_path} --output {onnx_path}
    print(f"\nONNX 模型已保存到: {onnx_path}")
except ImportError:
    print("\n提示: 安裝 tf2onnx 以使用 ONNX 轉換功能")
    print("pip install tf2onnx")

## 5. 模型優化技術

### 5.1 權重剪枝（Pruning）

In [ ]:
import tensorflow_model_optimization as tfmot

# 定義剪枝參數
pruning_params = {
    'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0,  # 初始稀疏度
        final_sparsity=0.5,    # 最終稀疏度（50% 的權重會被移除）
        begin_step=0,
        end_step=1000
    )
}

# 創建可剪枝模型
model_to_prune = create_model()
pruned_model = tfmot.sparsity.keras.prune_low_magnitude(model_to_prune, **pruning_params)

# 編譯
pruned_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 訓練（包含剪枝）
callbacks = [
    tfmot.sparsity.keras.UpdatePruningStep()  # 剪枝回調
]

print("訓練剪枝模型...")
pruned_model.fit(
    x_train[:5000], y_train[:5000],  # 使用部分數據以加快演示
    epochs=3,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

# 移除剪枝包裝器並保存
final_pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)
pruned_model_path = os.path.join(model_dir, 'pruned_model.h5')
final_pruned_model.save(pruned_model_path)

print(f"\n剪枝模型已保存")

### 5.2 知識蒸餾（Knowledge Distillation）

In [ ]:
class Distiller(keras.Model):
    """知識蒸餾訓練器"""
    
    def __init__(self, student, teacher):
        super().__init__()
        self.teacher = teacher
        self.student = student
    
    def compile(self, optimizer, metrics, student_loss_fn, distillation_loss_fn, alpha=0.1, temperature=3):
        super().compile(optimizer=optimizer, metrics=metrics)
        self.student_loss_fn = student_loss_fn
        self.distillation_loss_fn = distillation_loss_fn
        self.alpha = alpha  # 學生損失權重
        self.temperature = temperature  # 蒸餾溫度
    
    def train_step(self, data):
        x, y = data
        
        # 教師模型預測
        teacher_predictions = self.teacher(x, training=False)
        
        with tf.GradientTape() as tape:
            # 學生模型預測
            student_predictions = self.student(x, training=True)
            
            # 計算損失
            student_loss = self.student_loss_fn(y, student_predictions)
            distillation_loss = self.distillation_loss_fn(
                tf.nn.softmax(teacher_predictions / self.temperature, axis=1),
                tf.nn.softmax(student_predictions / self.temperature, axis=1)
            )
            total_loss = self.alpha * student_loss + (1 - self.alpha) * distillation_loss
        
        # 更新權重
        trainable_vars = self.student.trainable_variables
        gradients = tape.gradient(total_loss, trainable_vars)
        self.optimizer.apply_gradients(zip(gradients, trainable_vars))
        
        # 更新指標
        self.compiled_metrics.update_state(y, student_predictions)
        
        return {m.name: m.result() for m in self.metrics}

# 教師模型（已訓練的大模型）
teacher = model

# 學生模型（更小的模型）
student = keras.Sequential([
    keras.layers.Flatten(input_shape=(28, 28)),
    keras.layers.Dense(32, activation='relu'),  # 更少的神經元
    keras.layers.Dense(10, activation='softmax')
])

# 創建蒸餾器
distiller = Distiller(student=student, teacher=teacher)
distiller.compile(
    optimizer=keras.optimizers.Adam(),
    metrics=['accuracy'],
    student_loss_fn=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    distillation_loss_fn=keras.losses.KLDivergence(),
    alpha=0.1,
    temperature=3
)

# 訓練學生模型
print("知識蒸餾訓練...")
distiller.fit(x_train[:5000], y_train[:5000], epochs=3, verbose=1)

# 比較模型大小
teacher.save(os.path.join(model_dir, 'teacher.h5'))
student.save(os.path.join(model_dir, 'student.h5'))

teacher_size = os.path.getsize(os.path.join(model_dir, 'teacher.h5')) / 1024
student_size = os.path.getsize(os.path.join(model_dir, 'student.h5')) / 1024

print(f"\n教師模型大小: {teacher_size:.2f} KB")
print(f"學生模型大小: {student_size:.2f} KB")
print(f"大小減少: {teacher_size/student_size:.2f}x")

## 6. 部署策略

### 6.1 TensorFlow Serving（服務器端）

In [ ]:
# 為 TF Serving 準備模型
# 模型需要版本號
serving_model_dir = os.path.join(model_dir, 'tf_serving', '1')  # 版本 1
model.save(serving_model_dir)

print(f"TF Serving 模型已保存到: {serving_model_dir}")

# Docker 啟動 TF Serving 的命令
print("\n使用 Docker 啟動 TF Serving:")
print(f"""docker run -p 8501:8501 \\
  --mount type=bind,source={os.path.dirname(serving_model_dir)},target=/models/my_model \\
  -e MODEL_NAME=my_model \\
  -t tensorflow/serving
""")

# 客戶端請求示例
print("\n發送預測請求（Python）:")
print("""
import requests
import json

data = json.dumps({
    "signature_name": "serving_default",
    "instances": x_test[:3].tolist()
})

headers = {"content-type": "application/json"}
response = requests.post(
    'http://localhost:8501/v1/models/my_model:predict',
    data=data,
    headers=headers
)

predictions = json.loads(response.text)['predictions']
""")

### 6.2 TensorFlow.js（瀏覽器端）

In [ ]:
# 安裝 tensorflowjs（如果需要）
# !pip install tensorflowjs

# 轉換為 TF.js 格式
tfjs_dir = os.path.join(model_dir, 'tfjs_model')

print("轉換為 TF.js 的命令:")
print(f"tensorflowjs_converter --input_format=keras {h5_model_path} {tfjs_dir}")

# 實際轉換（如果已安裝 tensorflowjs）
try:
    !tensorflowjs_converter --input_format=keras {h5_model_path} {tfjs_dir}
    print(f"\nTF.js 模型已保存到: {tfjs_dir}")
    !ls -lh {tfjs_dir}
except:
    print("\n提示: 安裝 tensorflowjs 以使用此功能")
    print("pip install tensorflowjs")

# JavaScript 使用示例
print("\n在瀏覽器中使用（JavaScript）:")
print("""
<script src="https://cdn.jsdelivr.net/npm/@tensorflow/tfjs"></script>
<script>
  async function loadAndPredict() {
    // 載入模型
    const model = await tf.loadLayersModel('path/to/model.json');
    
    // 準備輸入數據
    const input = tf.tensor2d([[...your_data...]]);
    
    // 預測
    const prediction = model.predict(input);
    prediction.print();
  }
  
  loadAndPredict();
</script>
""")

### 6.3 移動端部署（Android/iOS）

In [ ]:
# Android 使用示例（Kotlin）
print("Android 集成（Kotlin）:")
print("""
// 1. 添加依賴
dependencies {
    implementation 'org.tensorflow:tensorflow-lite:2.13.0'
}

// 2. 載入模型
val model = Interpreter(loadModelFile())

// 3. 準備輸入
val input = Array(1) { FloatArray(784) }
// ... 填充數據 ...

// 4. 執行推理
val output = Array(1) { FloatArray(10) }
model.run(input, output)

// 5. 處理結果
val prediction = output[0].indices.maxByOrNull { output[0][it] } ?: -1
""")

print("\niOS 集成（Swift）:")
print("""
// 1. 導入框架
import TensorFlowLite

// 2. 載入模型
guard let modelPath = Bundle.main.path(forResource: "model", ofType: "tflite") else {
    return
}
let interpreter = try Interpreter(modelPath: modelPath)

// 3. 分配張量
try interpreter.allocateTensors()

// 4. 準備輸入
var inputData = Data()
// ... 填充數據 ...
try interpreter.copy(inputData, toInputAt: 0)

// 5. 執行推理
try interpreter.invoke()

// 6. 獲取輸出
let outputTensor = try interpreter.output(at: 0)
""")

## 7. 版本管理與實驗追蹤

### 7.1 使用 MLflow

In [ ]:
# 安裝 MLflow（如果需要）
# !pip install mlflow

try:
    import mlflow
    import mlflow.tensorflow
    
    # 開始實驗
    mlflow.set_experiment("mnist_experiments")
    
    with mlflow.start_run():
        # 記錄參數
        mlflow.log_param("epochs", 5)
        mlflow.log_param("batch_size", 128)
        mlflow.log_param("optimizer", "adam")
        
        # 訓練模型
        model = create_model()
        history = model.fit(x_train[:1000], y_train[:1000], epochs=2, verbose=0)
        
        # 記錄指標
        mlflow.log_metric("accuracy", history.history['accuracy'][-1])
        mlflow.log_metric("loss", history.history['loss'][-1])
        
        # 保存模型
        mlflow.tensorflow.log_model(model, "model")
        
        print("實驗已記錄到 MLflow")
        print("\n啟動 MLflow UI:")
        print("mlflow ui")
        print("訪問 http://localhost:5000 查看實驗")
        
except ImportError:
    print("安裝 MLflow:")
    print("pip install mlflow")

## 8. 最佳實踐總結

### ✅ 模型保存最佳實踐

1. **生產環境：** 使用 SavedModel 格式
2. **實驗階段：** 使用 ModelCheckpoint 定期保存
3. **移動端：** 轉換為 TFLite 並量化
4. **跨平台：** 考慮 ONNX 格式
5. **版本管理：** 使用語義化版本號（例如：model_v1.2.0）

### 📋 部署前檢查清單

- ✅ 模型性能已驗證（準確率、延遲）
- ✅ 模型已優化（剪枝、量化）
- ✅ 輸入/輸出格式已文檔化
- ✅ 版本號已正確設置
- ✅ 測試數據已準備
- ✅ 回滾策略已制定

### 🎯 選擇保存格式的決策樹

```
需要部署嗎？
├─ 是
│  ├─ 服務器端 → SavedModel + TF Serving
│  ├─ 移動端 → TFLite + 量化
│  ├─ 瀏覽器 → TensorFlow.js
│  └─ 跨框架 → ONNX
└─ 否（僅訓練/實驗）
   ├─ 需要架構 → SavedModel 或 HDF5
   └─ 僅權重 → Checkpoint
```

### ⚠️ 常見錯誤

1. **忘記保存訓練配置**
   ```python
   # ❌ 只保存權重，丟失架構
   model.save_weights('weights.h5')
   
   # ✅ 保存完整模型
   model.save('complete_model')
   ```

2. **版本不匹配**
   ```python
   # 記錄 TensorFlow 版本
   with open('model_info.txt', 'w') as f:
       f.write(f"TensorFlow: {tf.__version__}\n")
       f.write(f"Training date: {datetime.now()}\n")
   ```

3. **未測試載入的模型**
   ```python
   # ✅ 總是驗證載入的模型
   loaded_model = keras.models.load_model('model')
   test_loss, test_acc = loaded_model.evaluate(x_test, y_test)
   assert test_acc > 0.9, "載入的模型性能下降!"
   ```

---

## 🎓 進一步學習

- [TensorFlow 模型保存指南](https://www.tensorflow.org/guide/keras/save_and_serialize)
- [TensorFlow Lite 指南](https://www.tensorflow.org/lite/guide)
- [TensorFlow Serving 文檔](https://www.tensorflow.org/tfx/guide/serving)
- [TensorFlow Model Optimization](https://www.tensorflow.org/model_optimization)

---

## 📝 練習建議

1. 訓練一個模型並嘗試所有保存格式
2. 實作自動保存最佳模型的 callback
3. 將模型轉換為 TFLite 並比較性能
4. 實作簡單的 REST API 服務模型
5. 使用 MLflow 追蹤多個實驗